# Does the Grassmann geometry matter?

An ablation of *Attention Is Not What You Need: Grassmann Flows* (arXiv 2512.19428) on Wikitext-2, for a Colab **L4** GPU.

Seven ~12.6M-parameter language models, identical except for the token mixer:

| mixer | what it mixes |
|---|---|
| `grassmann` | the paper's layer: Plücker coordinates (antisymmetric product) of pairs (t, t−Δ), Δ ∈ {1,2,4,8,12,16} |
| `sym` | same layer, symmetric product (no orientation) |
| `outer` | same layer, full product |
| `concat` | same layer, no product: just the two vectors side by side |
| `hyena` | Hyena operator with a 16-step explicit filter |
| `swa` | sliding-window attention, window 16 |
| `attention` | full causal attention (the transformer baseline) |

The paper-style variants and the windowed baselines all see exactly 16 tokens back (checked by the tests), and every model is within 3% of the attention baseline's parameter count.

**Plan:** tests → data → smoke run (measures speed on this GPU) → learning-rate sweep (1 seed) → main runs (3 seeds at each mixer's best LR) → table with 95% confidence intervals and paired same-seed comparisons.

Results are written to Google Drive, and every run cell skips runs that already finished, so after a disconnect just re-run the cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Paths and Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO    = '/content/grassman'
DATA    = '/content/data'
OUTDIR  = '/content/drive/MyDrive/grassman/results'
RUNS    = f'{OUTDIR}/runs.jsonl'      # one JSON line per finished run
os.makedirs(OUTDIR, exist_ok=True)
print('results ->', RUNS)

## 2. Get the code

Either set `REPO_URL` to the GitHub repo, or leave it empty and upload `grassman.zip` (made on the Desktop) when asked.

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<you>/grassman.git'

import shutil, zipfile
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if REPO_URL:
    !git clone -q {REPO_URL} {REPO}
else:
    from google.colab import files
    up = files.upload()                       # pick grassman.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd {REPO}
!git log --oneline -1 2>/dev/null || echo "(no git history in this copy)"

In [ ]:
!pip install -q datasets transformers

## 3. Tests

84 tests, on CPU, about a minute. They check the Plücker maths (definition, 3-D cross product, sign flip, plane invariance), that **no mixer reads future tokens** (the paper's equations as written pair t with t+Δ, which would leak), that each windowed mixer reaches exactly 16 tokens, parameter matching, determinism, and the training/analysis code end to end. Do not continue if anything fails.

In [ ]:
!python -m pytest -q

## 4. Data: Wikitext-2-raw, bert-base-uncased WordPiece (as in the paper)

In [ ]:
!python -m grassman.data --data-dir {DATA}

## 5. Smoke run: speed on this GPU

200 steps per mixer. Prints throughput and the **estimated time of one full 30-epoch run** (≈ 69M training tokens), so you can budget the sweep and main runs before starting them.

In [ ]:
import json, subprocess, sys

def train(mixer, seed=0, lr=1e-3, epochs=30, tag='main', out=RUNS, extra=()):
    cmd = [sys.executable, '-m', 'grassman.train', '--mixer', mixer, '--seed', str(seed),
           '--lr', str(lr), '--epochs', str(epochs), '--tag', tag, '--data-dir', DATA,
           '--out', out, '--skip-existing', *extra]
    print('>>', ' '.join(cmd[3:]), flush=True)
    subprocess.run(cmd, cwd=REPO, check=True)

MIXERS = ['grassmann', 'sym', 'outer', 'concat', 'hyena', 'swa', 'attention']
SMOKE = '/content/smoke.jsonl'
if os.path.exists(SMOKE):
    os.remove(SMOKE)
for m in MIXERS:
    train(m, epochs=1, tag='smoke', out=SMOKE, extra=['--max-steps', '200'])

full_tokens = 30 * 2_303_695
print(f"\n{'mixer':<10} {'params':>8} {'tok/s':>10} {'est. 30-epoch run':>18}")
for line in open(SMOKE):
    r = json.loads(line)
    print(f"{r['config']['mixer']:<10} {r['params']/1e6:7.2f}M {r['tokens_per_s']:10,.0f} "
          f"{full_tokens / r['tokens_per_s'] / 60:15.1f} min")

## 6. Stage 1: learning-rate sweep

The paper does not report its optimizer settings, so each mixer gets its own best learning rate (one seed, 10 epochs), chosen on **validation** perplexity only. This prevents any model from losing just because of a bad LR.

In [ ]:
LRS = [3e-4, 1e-3, 3e-3]
SWEEP_EPOCHS = 10
for m in MIXERS:
    for lr in LRS:
        train(m, seed=0, lr=lr, epochs=SWEEP_EPOCHS, tag='sweep')

In [ ]:
best = json.loads(subprocess.check_output(
    [sys.executable, '-m', 'grassman.analyze', RUNS, '--tag', 'sweep', '--best-lr'], cwd=REPO, text=True))
print(best)
!python -m grassman.analyze {RUNS} --tag sweep

## 7. Stage 2: main runs

3 seeds × 7 mixers, 30 epochs (as in the paper), each at its best LR. Test perplexity is taken at the epoch with the best validation perplexity.

For the headline comparison (`grassmann` vs `sym`) consider raising to 5 seeds if the confidence interval in step 8 straddles zero.

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 30
for seed in SEEDS:                 # seed-major: a disconnect still leaves complete seeds
    for m in MIXERS:
        train(m, seed=seed, lr=best[m], epochs=EPOCHS, tag='main')

Optional: the paper L2-normalises the Plücker vector. This checks whether that matters (shows up as `grassmann-raw`).

In [ ]:
for seed in SEEDS:
    train('grassmann', seed=seed, lr=best['grassmann'], epochs=EPOCHS, tag='main', extra=['--no-normalize'])

## 8. Results

In [ ]:
!python -m grassman.analyze {RUNS} --tag main

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

curves = defaultdict(list)
for line in open(RUNS):
    r = json.loads(line)
    if r['tag'] != 'main':
        continue
    name = r['config']['mixer'] + ('' if r['config']['normalize'] else '-raw')
    curves[name].append([h['val_ppl'] for h in r['history']])

fig, ax = plt.subplots(figsize=(8, 5))
for name, runs in sorted(curves.items()):
    n = min(len(c) for c in runs)
    mean = [sum(c[i] for c in runs) / len(runs) for i in range(n)]
    ax.plot(range(1, n + 1), mean, label=f'{name} ({len(runs)} seeds)')
ax.set_xlabel('epoch'); ax.set_ylabel('validation perplexity (mean over seeds)')
ax.set_ylim(top=600); ax.legend(); ax.grid(alpha=0.3)
plt.savefig(f'{OUTDIR}/val_curves.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read the table

- **Reproduction check.** `grassmann` vs `attention` in the "vs attention" column: the paper reports 10–15% worse, an independent reproduction (github.com/Infatoshi/grassmann-flows) 22.6% worse.
- **Does the geometry matter?** Look at the paired rows for `sym`, `outer` and `concat`. If their difference is positive and the 95% CI excludes 0, the antisymmetric (Plücker) features beat that alternative at the same feature budget. If the CIs include 0, the geometry is not what makes the layer work.
- **Is it better than simpler local mixers?** The same for `hyena` and `swa`, which see the same 16 tokens.
- With 3 seeds the intervals are wide; "excludes 0" is evidence, not proof. Report every number, including the ones that go against the paper.